In [1]:
import os
import cv2
import random
import numpy as np
import joblib
from skimage.feature import hog
from tqdm import tqdm

In [3]:
# --- CONFIGURATION ---
# Pointing directly to your new single dataset folder
DATASET_PATH = r"Dataset\\asl_alphabet_train"

MAX_IMAGES_PER_CLASS = 500 # Sweet spot for SVM processing time and accuracy
IMAGE_SIZE = (128, 128) # High-def resolution for U vs V distinction
OUTPUT_FILE = "final_hog_features.joblib"

In [6]:
X_features = []
y_labels = []

# --- EXTRACTION LOOP ---
print(f"\nProcessing dataset: {DATASET_PATH}")

if not os.path.exists(DATASET_PATH):
    print(f"Error: Could not find the folder '{DATASET_PATH}'. Please check the path.")
else:
    # Get all class folders (A-Z, 0-9)
    class_folders = [f for f in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, f))]
    
    for class_name in tqdm(class_folders, desc="Extracting HOG Features"):
        class_folder = os.path.join(DATASET_PATH, class_name)
        all_images = os.listdir(class_folder)
        
        # Randomly shuffle and slice to ensure diverse training data
        if len(all_images) > MAX_IMAGES_PER_CLASS:
            selected_images = random.sample(all_images, MAX_IMAGES_PER_CLASS)
        else:
            selected_images = all_images
            
        for image_name in selected_images:
            img_path = os.path.join(class_folder, image_name)
            img = cv2.imread(img_path)
            
            if img is not None:
                # 1. Grayscale Conversion
                gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                
                # 2. High-Def Resize
                resized_img = cv2.resize(gray_img, IMAGE_SIZE)
                
                # 3. HOG Feature Extraction
                features_hog = hog(resized_img, 
                                   orientations=9, 
                                   pixels_per_cell=(16, 16), # Optimized for 128x128
                                   cells_per_block=(2, 2), 
                                   block_norm='L2-Hys', 
                                   visualize=False)
    
                X_features.append(features_hog)
                y_labels.append(class_name.upper())
# --- PACKAGING AND SAVING ---
    print("\nExtraction complete! Converting to arrays...")
    X = np.array(X_features)
    y = np.array(y_labels)

    data_package = {
        "X": X,
        "y": y
    }

    print(f"Saving data to {OUTPUT_FILE}...")
    joblib.dump(data_package, OUTPUT_FILE, compress=9)

    print("-" * 40)
    print(" SUCCESS!")
    print(f" Total Images Processed: {len(y)}")
    print(f" Final Feature Shape: {X.shape}")
    print("-" * 40)


Processing dataset: Dataset\\asl_alphabet_train


Extracting HOG Features: 100%|██████████| 29/29 [04:45<00:00,  9.83s/it]



Extraction complete! Converting to arrays...
Saving data to final_hog_features.joblib...
----------------------------------------
 SUCCESS!
 Total Images Processed: 14500
 Final Feature Shape: (14500, 1764)
----------------------------------------
